# MinerU Kaggle — 2×T4 + Resume + mỗi PDF một thư mục riêng

Notebook này chạy trực tiếp trên Kaggle với **2 GPU T4**.

Mỗi PDF được lưu thành một thư mục riêng:

```text
<giữ nguyên cây input>/
└── EN_EN-17_Tutorial on DNN for Intelligent Systems/
    ├── images/
    ├── EN_EN-17_Tutorial on DNN for Intelligent Systems.md
    ├── _SUCCESS.json
    ├── middle_json.json
    ├── mineru_original.md.txt
    ├── model_output.json
    └── structured_content.json
```

Hai worker độc lập cùng lấy PDF từ một queue chung:

```text
GPU 0 -> Worker 0 -> PDF tiếp theo
GPU 1 -> Worker 1 -> PDF tiếp theo
```

Một document chỉ được xem là hoàn tất khi có cả file Markdown chính và `_SUCCESS.json`.
Nếu output cũ vẫn còn khi chạy lại, document hoàn tất sẽ được **SKIP**.

> Với batch dài, nên chạy bằng **Save Version → Save & Run All** thay vì treo Draft session.

In [ ]:
# =========================
# 1. CẤU HÌNH
# =========================
from pathlib import Path
import os

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

OUTPUT_ROOT = WORK_ROOT / 'mineru_rag_output'
STATE_ROOT = WORK_ROOT / 'mineru_state'
JOBS_ROOT = STATE_ROOT / 'jobs'
TEMP_ROOT = WORK_ROOT / 'mineru_temp'
WORKER_SCRIPT = WORK_ROOT / 'mineru_2gpu_worker.py'
FINAL_ZIP = WORK_ROOT / 'mineru_rag_output.zip'

TIER = 'standard'
OCR_MODE = 'auto'
MAX_FILES = None          # None = chạy toàn bộ PDF
MAX_RETRIES = 2           # retry thêm tối đa 2 lần nếu một PDF lỗi
CLEAN_OUTPUT = False      # PHẢI để False để resume
AUTO_STRIP_SINGLE_DATASET_ROOT = True
GPU_IDS = [0, 1]

print('INPUT_ROOT :', INPUT_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('STATE_ROOT :', STATE_ROOT)
print('GPU_IDS    :', GPU_IDS)

In [ ]:
# =========================
# 2. KIỂM TRA 2 GPU
# =========================
import subprocess

subprocess.run(['nvidia-smi'], check=False)

try:
    import torch
    print('\nTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    print('Visible GPU count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(
            f'GPU {i}:', torch.cuda.get_device_name(i),
            '| VRAM:', round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 2), 'GB'
        )
except Exception as e:
    print('Torch check error:', repr(e))

In [ ]:
# =========================
# 3. CÀI MINERU
# =========================
# Nếu session hiện tại đã cài MinerU và chạy tốt, có thể skip cell này.
import sys, subprocess

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-U', 'mineru[full]>=4.0,<5'],
    check=True,
)
print('MinerU installed.')

In [ ]:
# =========================
# 4. KIỂM TRA / TẢI MODEL
# =========================
import os, subprocess

os.environ['MINERU_MODEL_SMALL_BACKEND'] = 'torch'
os.environ['MINERU_MODEL_VLM_ENGINE'] = 'vllm'
os.environ['MINERU_MODEL_SOURCE'] = 'auto'

subprocess.run(['mineru', 'version'], check=False)
subprocess.run(['mineru-kit', 'models', 'show'], check=False)

subprocess.run([
    'mineru-kit', 'models', 'download',
    '--tier', TIER,
    '--small-backend', 'torch',
    '--vlm-engine', 'vllm',
], check=True)

subprocess.run([
    'mineru-kit', 'models', 'verify',
    '--tier', TIER,
    '--small-backend', 'torch',
    '--vlm-engine', 'vllm',
], check=True)

In [ ]:
# =========================
# 5. QUÉT TOÀN BỘ PDF NHIỀU CẤP
# =========================
from pathlib import Path

all_pdf_files = sorted(
    p for p in INPUT_ROOT.rglob('*')
    if p.is_file() and p.suffix.lower() == '.pdf'
)

if not all_pdf_files:
    raise RuntimeError('Không tìm thấy PDF trong /kaggle/input')


def detect_mirror_root(files):
    if not AUTO_STRIP_SINGLE_DATASET_ROOT:
        return INPUT_ROOT
    first_parts = set()
    for p in files:
        rel = p.relative_to(INPUT_ROOT)
        if rel.parts:
            first_parts.add(rel.parts[0])
    if len(first_parts) == 1:
        candidate = INPUT_ROOT / next(iter(first_parts))
        if candidate.is_dir():
            return candidate
    return INPUT_ROOT


MIRROR_ROOT = detect_mirror_root(all_pdf_files)
pdf_files = all_pdf_files if MAX_FILES is None else all_pdf_files[:MAX_FILES]

print('MIRROR_ROOT :', MIRROR_ROOT)
print('Total PDF   :', len(all_pdf_files))
print('Will process:', len(pdf_files))

for i, p in enumerate(pdf_files[:30], 1):
    try:
        rel = p.relative_to(MIRROR_ROOT)
    except Exception:
        rel = p.relative_to(INPUT_ROOT)
    print(f'{i:03d}. {rel}')
if len(pdf_files) > 30:
    print(f'... +{len(pdf_files)-30} file(s)')

In [ ]:
# =========================
# 6. TẠO WORKER SCRIPT
# =========================
WORKER_CODE = 'import argparse\nimport gc\nimport json\nimport os\nimport re\nimport shutil\nimport time\nimport traceback\nfrom copy import deepcopy\nfrom pathlib import Path\n\nparser = argparse.ArgumentParser()\nparser.add_argument(\'--gpu\', type=int, required=True)\nparser.add_argument(\'--jobs\', required=True)\nparser.add_argument(\'--output\', required=True)\nparser.add_argument(\'--temp\', required=True)\nparser.add_argument(\'--tier\', default=\'standard\')\nparser.add_argument(\'--ocr-mode\', default=\'auto\')\nparser.add_argument(\'--max-retries\', type=int, default=2)\nargs = parser.parse_args()\n\n# Must be set before importing MinerU / torch / vLLM.\nos.environ[\'CUDA_VISIBLE_DEVICES\'] = str(args.gpu)\nos.environ[\'MINERU_MODEL_SMALL_BACKEND\'] = \'torch\'\nos.environ[\'MINERU_MODEL_VLM_ENGINE\'] = \'vllm\'\nos.environ.setdefault(\'MINERU_MODEL_SOURCE\', \'auto\')\n\nJOBS_ROOT = Path(args.jobs)\nOUTPUT_ROOT = Path(args.output)\nTEMP_ROOT = Path(args.temp) / f\'gpu_{args.gpu}\'\nLOG_FILE = JOBS_ROOT.parent / f\'gpu_{args.gpu}.jsonl\'\nTEMP_ROOT.mkdir(parents=True, exist_ok=True)\n\nfrom mineru.parser import parse\nfrom mineru.parser.writer import FileBasedDataWriter\nfrom mineru.render import render, RenderFormat\nfrom mineru.types import MiddleJson\n\n\ndef log_event(event):\n    payload = {\'time\': time.strftime(\'%Y-%m-%d %H:%M:%S\'), \'gpu\': args.gpu, **event}\n    with LOG_FILE.open(\'a\', encoding=\'utf-8\') as f:\n        f.write(json.dumps(payload, ensure_ascii=False) + \'\\n\')\n        f.flush()\n        os.fsync(f.fileno())\n\n\ndef safe_asset_name(text, max_len=120):\n    text = text.strip()\n    text = re.sub(r\'[\\\\/:*?"<>|]+\', \'_\', text)\n    text = re.sub(r\'\\s+\', \'_\', text).strip(\' ._\')\n    return (text[:max_len].rstrip(\'._\') if len(text) > max_len else text) or \'document\'\n\n\ndef load_middle(data):\n    if hasattr(MiddleJson, \'model_validate\'):\n        return MiddleJson.model_validate(data)\n    if hasattr(MiddleJson, \'parse_obj\'):\n        return MiddleJson.parse_obj(data)\n    return MiddleJson(**data)\n\n\ndef one_page_middle(full_data, page_data):\n    one = deepcopy(full_data)\n    page_idx = page_data[\'page_idx\']\n    one[\'pages\'] = [deepcopy(page_data)]\n    one[\'is_full_document\'] = False\n    ext = one.get(\'extensions\')\n    if isinstance(ext, dict):\n        layout = ext.get(\'docvortex_layout\')\n        if isinstance(layout, dict) and isinstance(layout.get(\'pages\'), list):\n            layout[\'pages\'] = [p for p in layout[\'pages\'] if isinstance(p, dict) and p.get(\'page_idx\') == page_idx]\n    return load_middle(one)\n\n\ndef marked_markdown(middle_path):\n    full_data = json.loads(middle_path.read_text(encoding=\'utf-8\'))\n    parts = []\n    for page in full_data.get(\'pages\', []):\n        page_num = int(page[\'page_idx\']) + 1\n        page_md = render(one_page_middle(full_data, page), RenderFormat.MARKDOWN)\n        page_md = \'\' if page_md is None else str(page_md).strip()\n        marker = f\'<!-- page: {page_num} -->\'\n        parts.append(f\'{marker}\\n\\n{page_md}\' if page_md else marker)\n    return \'\\n\\n\'.join(parts).rstrip() + \'\\n\'\n\n\ndef find_file(root, filename):\n    direct = root / filename\n    if direct.exists():\n        return direct\n    found = list(root.rglob(filename))\n    if not found:\n        raise FileNotFoundError(f\'Cannot find {filename} under {root}\')\n    return found[0]\n\n\ndef rewrite_image_links(text, asset_dir_name):\n    prefix = f\'{asset_dir_name}/images/\'\n    text = re.sub(r\'(\\]\\()images/\', lambda m: m.group(1) + prefix, text)\n    text = re.sub(r\'(src=["\\\'])images/\', lambda m: m.group(1) + prefix, text, flags=re.I)\n    return text\n\n\ndef output_paths(job):\n    rel = Path(job[\'relative_path\'])\n    doc_dir = OUTPUT_ROOT / rel.parent / rel.stem\n    final_md = doc_dir / f"{rel.stem}.md"\n    success = doc_dir / \'_SUCCESS.json\'\n    return doc_dir, final_md, success\n\n\ndef is_completed(job):\n    doc_dir, final_md, success = output_paths(job)\n    if not doc_dir.exists() or not final_md.exists() or not success.exists():\n        return False\n    try:\n        return json.loads(success.read_text(encoding=\'utf-8\')).get(\'status\') == \'ok\'\n    except Exception:\n        return False\n\n\ndef atomic_write_text(path, text):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + \'.partial\')\n    tmp.write_text(text, encoding=\'utf-8\')\n    os.replace(tmp, path)\n\n\ndef process_job(job):\n    pdf_path = Path(job[\'source_path\'])\n    rel = Path(job[\'relative_path\'])\n    job_id = job[\'job_id\']\n    final_doc_dir, final_md, _ = output_paths(job)\n    final_doc_dir.parent.mkdir(parents=True, exist_ok=True)\n\n    work_dir = TEMP_ROOT / f\'{job_id}.work\'\n    staged_doc = TEMP_ROOT / f\'{job_id}.document.partial\'\n\n    for p in (work_dir, staged_doc):\n        if p.exists():\n            shutil.rmtree(p)\n    work_dir.mkdir(parents=True, exist_ok=True)\n\n    started = time.perf_counter()\n\n    # Parse PDF đúng một lần.\n    result = parse(\n        str(pdf_path),\n        tier=args.tier,\n        ocr_mode=args.ocr_mode,\n        image_analysis=True,\n        page_range=\'all\',\n    )\n\n    # Lưu raw output MinerU vào thư mục tạm.\n    result.save(FileBasedDataWriter(str(work_dir)))\n\n    middle_path = find_file(work_dir, \'middle_json.json\')\n    middle_data = json.loads(middle_path.read_text(encoding=\'utf-8\'))\n    page_count = len(middle_data.get(\'pages\', []))\n\n    # Markdown có marker trang. Ảnh giữ đường dẫn images/... vì\n    # file .md và thư mục images nằm cùng document folder.\n    md = marked_markdown(middle_path)\n\n    # Stage toàn bộ document folder trước khi commit.\n    shutil.copytree(work_dir, staged_doc)\n\n    # Đổi markdown.md gốc để rglob(\'*.md\') không lấy nhầm file debug.\n    try:\n        original_md = find_file(staged_doc, \'markdown.md\')\n        original_md.rename(original_md.with_name(\'mineru_original.md.txt\'))\n    except FileNotFoundError:\n        pass\n\n    staged_md = staged_doc / f\'{rel.stem}.md\'\n    staged_md.write_text(md, encoding=\'utf-8\')\n\n    info = {\n        \'status\': \'ok\',\n        \'gpu\': args.gpu,\n        \'source_pdf\': str(pdf_path),\n        \'source_relative\': str(rel),\n        \'output_relative\': str(rel.parent / rel.stem),\n        \'markdown\': f\'{rel.stem}.md\',\n        \'page_count\': page_count,\n        \'seconds\': round(time.perf_counter() - started, 2),\n        \'finished_at\': time.strftime(\'%Y-%m-%d %H:%M:%S\'),\n    }\n\n    # _SUCCESS chỉ xuất hiện sau khi toàn bộ document đã stage xong.\n    (staged_doc / \'_SUCCESS.json\').write_text(\n        json.dumps(info, ensure_ascii=False, indent=2),\n        encoding=\'utf-8\',\n    )\n\n    # Commit document folder. Nếu có output dở cũ, thay thế nó.\n    if final_doc_dir.exists():\n        shutil.rmtree(final_doc_dir)\n    os.replace(staged_doc, final_doc_dir)\n\n    if work_dir.exists():\n        shutil.rmtree(work_dir)\n\n    del result\n    gc.collect()\n    try:\n        import torch\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n    except Exception:\n        pass\n\n    return info\n\n\ndef claim_next():\n    for pending in sorted(JOBS_ROOT.glob(\'*.pending.json\')):\n        running = pending.with_name(pending.name.replace(\'.pending.json\', f\'.gpu{args.gpu}.running.json\'))\n        try:\n            os.replace(pending, running)\n            return running\n        except (FileNotFoundError, OSError):\n            continue\n    return None\n\n\ndef mark_done(running, job, result, skipped=False):\n    done = running.with_name(re.sub(r\'\\.gpu\\d+\\.running\\.json$\', \'.done.json\', running.name))\n    done.write_text(json.dumps({**job, \'worker_gpu\': args.gpu, \'result\': result, \'skipped\': skipped}, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    if running.exists():\n        running.unlink()\n\n\ndef mark_failed(running, job, exc):\n    attempts = int(job.get(\'attempts\', 0)) + 1\n    job[\'attempts\'] = attempts\n    job[\'last_error\'] = repr(exc)\n    job[\'last_traceback\'] = traceback.format_exc()\n    if attempts <= args.max_retries:\n        target = running.with_name(re.sub(r\'\\.gpu\\d+\\.running\\.json$\', \'.pending.json\', running.name))\n        status = \'retry\'\n    else:\n        target = running.with_name(re.sub(r\'\\.gpu\\d+\\.running\\.json$\', \'.failed.json\', running.name))\n        status = \'failed\'\n    target.write_text(json.dumps(job, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    if running.exists():\n        running.unlink()\n    log_event({\'event\': status, \'job_id\': job.get(\'job_id\'), \'relative_path\': job.get(\'relative_path\'), \'attempts\': attempts, \'error\': repr(exc)})\n\n\ndef main():\n    log_event({\'event\': \'worker_start\'})\n    try:\n        import torch\n        log_event({\'event\': \'gpu_info\', \'cuda_available\': torch.cuda.is_available(), \'visible_count\': torch.cuda.device_count(), \'device_name\': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})\n    except Exception as e:\n        log_event({\'event\': \'gpu_info_error\', \'error\': repr(e)})\n\n    while True:\n        running = claim_next()\n        if running is None:\n            break\n        job = None\n        try:\n            job = json.loads(running.read_text(encoding=\'utf-8\'))\n            if is_completed(job):\n                mark_done(running, job, {\'status\': \'ok\', \'resume_skip\': True}, skipped=True)\n                log_event({\'event\': \'skip_completed\', \'job_id\': job[\'job_id\'], \'relative_path\': job[\'relative_path\']})\n                continue\n            log_event({\'event\': \'start\', \'job_id\': job[\'job_id\'], \'relative_path\': job[\'relative_path\']})\n            result = process_job(job)\n            mark_done(running, job, result)\n            log_event({\'event\': \'done\', \'job_id\': job[\'job_id\'], \'relative_path\': job[\'relative_path\'], \'pages\': result[\'page_count\'], \'seconds\': result[\'seconds\']})\n        except Exception as e:\n            if job is None:\n                job = {\'job_id\': running.stem, \'relative_path\': str(running), \'attempts\': 0}\n            mark_failed(running, job, e)\n\n    log_event({\'event\': \'worker_exit\'})\n\n\nif __name__ == \'__main__\':\n    main()\n'
WORKER_SCRIPT.write_text(WORKER_CODE, encoding='utf-8')
print('Worker script:', WORKER_SCRIPT)
print('Size:', WORKER_SCRIPT.stat().st_size, 'bytes')


In [ ]:
# =========================
# 7. TẠO JOB QUEUE TỪ OUTPUT HIỆN CÓ
# =========================
import hashlib, json, shutil

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
STATE_ROOT.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

if CLEAN_OUTPUT:
    raise RuntimeError('Hãy để CLEAN_OUTPUT=False để không xóa checkpoint cũ.')

# Queue chỉ là state tạm. Mỗi lần chạy, tạo lại từ output thật.
if JOBS_ROOT.exists():
    shutil.rmtree(JOBS_ROOT)
JOBS_ROOT.mkdir(parents=True, exist_ok=True)


def source_relative_path(pdf_path):
    try:
        return pdf_path.relative_to(MIRROR_ROOT)
    except ValueError:
        return pdf_path.relative_to(INPUT_ROOT)


def job_id_for(pdf_path):
    rel = str(source_relative_path(pdf_path))
    return hashlib.sha1(rel.encode('utf-8')).hexdigest()[:16]


def output_paths_for_pdf(pdf_path):
    rel = source_relative_path(pdf_path)
    doc_dir = OUTPUT_ROOT / rel.parent / rel.stem
    md_path = doc_dir / f'{rel.stem}.md'
    success_path = doc_dir / '_SUCCESS.json'
    return rel, doc_dir, md_path, success_path


def output_is_complete(pdf_path):
    rel, doc_dir, md_path, success_path = output_paths_for_pdf(pdf_path)
    if not md_path.exists() or not success_path.exists():
        return False
    try:
        info = json.loads(success_path.read_text(encoding='utf-8'))
        return info.get('status') == 'ok'
    except Exception:
        return False


already_completed = 0
created = 0

for pdf_path in pdf_files:
    rel = source_relative_path(pdf_path)
    jid = job_id_for(pdf_path)

    if output_is_complete(pdf_path):
        already_completed += 1
        continue

    payload = {
        'job_id': jid,
        'source_path': str(pdf_path),
        'relative_path': str(rel),
        'attempts': 0,
    }

    pending = JOBS_ROOT / f'{jid}.pending.json'
    pending.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    created += 1

print('Already completed :', already_completed)
print('New pending jobs  :', created)
print('Selected PDFs     :', len(pdf_files))

## 8. Chạy 2 GPU

Hai process được gán GPU riêng trước khi import MinerU/vLLM.
GPU nào xong trước sẽ claim PDF tiếp theo trong queue.

In [ ]:
# =========================
# 8. CHẠY 2 WORKER + MONITOR
# =========================
import os, subprocess, sys, time

env_base = os.environ.copy()
env_base['MINERU_MODEL_SMALL_BACKEND'] = 'torch'
env_base['MINERU_MODEL_VLM_ENGINE'] = 'vllm'
env_base['MINERU_MODEL_SOURCE'] = env_base.get('MINERU_MODEL_SOURCE', 'auto')

# Xóa log cũ của phiên hiện tại; checkpoint output không bị xóa.
for gpu in GPU_IDS:
    for name in [f'gpu_{gpu}.jsonl', f'gpu_{gpu}.stdout.log']:
        p = STATE_ROOT / name
        if p.exists():
            p.unlink()

processes = []
for gpu in GPU_IDS:
    stdout_path = STATE_ROOT / f'gpu_{gpu}.stdout.log'
    stdout_file = stdout_path.open('w', encoding='utf-8')
    cmd = [
        sys.executable, str(WORKER_SCRIPT),
        '--gpu', str(gpu),
        '--jobs', str(JOBS_ROOT),
        '--output', str(OUTPUT_ROOT),
        '--temp', str(TEMP_ROOT),
        '--tier', TIER,
        '--ocr-mode', OCR_MODE,
        '--max-retries', str(MAX_RETRIES),
    ]
    p = subprocess.Popen(cmd, stdout=stdout_file, stderr=subprocess.STDOUT, env=env_base)
    processes.append((gpu, p, stdout_file, stdout_path))
    print(f'Started GPU {gpu}: PID={p.pid}')

print('\nWorkers are running...')
last_print = 0

while True:
    alive = [p.poll() is None for _, p, _, _ in processes]
    now = time.time()

    if now - last_print >= 15:
        pending = len(list(JOBS_ROOT.glob('*.pending.json')))
        running = len(list(JOBS_ROOT.glob('*.gpu*.running.json')))
        done = len(list(JOBS_ROOT.glob('*.done.json')))
        failed = len(list(JOBS_ROOT.glob('*.failed.json')))
        print(f'[STATUS] pending={pending} | running={running} | done={done} | failed={failed}')
        subprocess.run([
            'nvidia-smi',
            '--query-gpu=index,name,memory.used,memory.total,utilization.gpu',
            '--format=csv,noheader,nounits',
        ], check=False)
        last_print = now

    if not any(alive):
        break
    time.sleep(3)

for gpu, p, stdout_file, stdout_path in processes:
    stdout_file.close()
    print(f'GPU {gpu} exit code: {p.returncode}')
    print(f'GPU {gpu} stdout   : {stdout_path}')

print('\nAll workers exited.')

In [ ]:
# =========================
# 9. TỔNG KẾT + MANIFEST
# =========================
import json

pending_files = list(JOBS_ROOT.glob('*.pending.json'))
running_files = list(JOBS_ROOT.glob('*.gpu*.running.json'))
done_files = list(JOBS_ROOT.glob('*.done.json'))
failed_files = list(JOBS_ROOT.glob('*.failed.json'))
md_files = list(OUTPUT_ROOT.rglob('*.md'))
success_files = list(OUTPUT_ROOT.rglob('_SUCCESS.json'))

failures = []
for path in failed_files:
    try:
        failures.append(json.loads(path.read_text(encoding='utf-8')))
    except Exception:
        pass

manifest = {
    'input_root': str(INPUT_ROOT),
    'mirror_root': str(MIRROR_ROOT),
    'tier': TIER,
    'ocr_mode': OCR_MODE,
    'gpu_ids': GPU_IDS,
    'pdf_count': len(pdf_files),
    'markdown_count': len(md_files),
    'success_marker_count': len(success_files),
    'pending_jobs': len(pending_files),
    'running_jobs': len(running_files),
    'done_jobs': len(done_files),
    'failed_jobs': len(failed_files),
    'failures': failures,
}

manifest_path = OUTPUT_ROOT / '_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(manifest, ensure_ascii=False, indent=2))

if pending_files or running_files or failed_files:
    print('\nWARNING: Chưa hoàn tất toàn bộ.')
else:
    print('\nSUCCESS: Không còn pending/running/failed job.')

In [ ]:
# =========================
# 10. XEM PHÂN BỐ CÔNG VIỆC 2 GPU
# =========================
import json

for gpu in GPU_IDS:
    print('\n' + '=' * 90)
    print(f'GPU {gpu}')
    print('=' * 90)
    log_path = STATE_ROOT / f'gpu_{gpu}.jsonl'
    if not log_path.exists():
        print('No log.')
        continue
    events = []
    for line in log_path.read_text(encoding='utf-8').splitlines():
        try:
            events.append(json.loads(line))
        except Exception:
            pass
    done_events = [x for x in events if x.get('event') == 'done']
    retry_events = [x for x in events if x.get('event') in {'retry', 'failed'}]
    print('Completed:', len(done_events))
    print('Retry/failed events:', len(retry_events))
    for item in done_events[-10:]:
        print(f"[OK] {item.get('relative_path')} | pages={item.get('pages')} | time={item.get('seconds')}s")

In [ ]:
# =========================
# 11. VALIDATE PAGE MARKERS
# =========================
import json, re

bad = []
checked = 0

for success_path in OUTPUT_ROOT.rglob('_SUCCESS.json'):
    try:
        info = json.loads(success_path.read_text(encoding='utf-8'))
        md_path = success_path.parent / info['markdown']

        if not md_path.exists():
            bad.append((str(md_path), 'missing markdown'))
            continue

        text = md_path.read_text(encoding='utf-8')
        markers = [int(x) for x in re.findall(r'<!-- page: (\d+) -->', text)]
        expected = list(range(1, int(info['page_count']) + 1))

        if markers != expected:
            bad.append((str(md_path), f'markers={len(markers)} expected={len(expected)}'))

        checked += 1

    except Exception as e:
        bad.append((str(success_path), repr(e)))

print('Checked:', checked)
print('Bad    :', len(bad))
for item in bad[:30]:
    print(item)

In [ ]:
# =========================
# 12. NÉN ZIP OUTPUT
# =========================
import shutil

if FINAL_ZIP.exists():
    FINAL_ZIP.unlink()

created = shutil.make_archive(
    str(FINAL_ZIP.with_suffix('')),
    'zip',
    root_dir=OUTPUT_ROOT,
)

print('ZIP :', created)
print('Size:', round(Path(created).stat().st_size / 1024**3, 3), 'GB')

# Cách chạy an toàn

## Trước khi chạy toàn bộ

1. Attach lại dataset PDF vào **Input**.
2. Chọn accelerator **2×T4**.
3. Giữ `CLEAN_OUTPUT = False`.
4. Notebook mặc định `MAX_FILES = None`, tức là xử lý toàn bộ PDF.
5. Nên chạy bằng **Save Version → Save & Run All** thay vì treo Draft session.

## Output mỗi PDF

```text
<giữ nguyên cây input>/
└── abc/
    ├── images/
    ├── abc.md
    ├── _SUCCESS.json
    ├── middle_json.json
    ├── mineru_original.md.txt
    ├── model_output.json
    └── structured_content.json
```

Markdown và `images/` ở cùng thư mục nên link `![](images/...)` giữ nguyên.

## Resume

Nếu filesystem/output cũ còn tồn tại khi chạy lại:

```text
abc/abc.md + abc/_SUCCESS.json  -> SKIP
chưa đủ hai file                 -> parse lại riêng PDF đó
```

Queue được tạo lại từ output thật mỗi lần chạy, nên state job cũ không làm mất PDF.

## Chunk RAG

Chỉ cần lấy Markdown chính:

```python
markdown_files = list(
    Path('/kaggle/working/mineru_rag_output').rglob('*.md')
)
```

`markdown.md` gốc của MinerU được đổi thành `mineru_original.md.txt`, nên không bị lấy nhầm.